# Listing Amazon Bedrock Foundation Models

A quick demo of calling Amazon Bedrock from Python with boto3.

This notebook builds up in a few steps:
1. Make the call and look at the response
2. See the response structure (matches the slide)
3. Print just the model names and IDs
4. Look "under the hood" at the actual HTTP request and response (like Chrome DevTools)

## 1. Call Bedrock

The `bedrock` client talks to the Bedrock control plane, which knows *about* the models (metadata).

In [ ]:
import boto3

bedrock = boto3.client("bedrock", region_name="us-east-1")
response = bedrock.list_foundation_models()

The last expression in a cell is rendered as output. A dict or list shows up as a collapsible tree you can expand and inspect.

In [ ]:
response["modelSummaries"]

## 2. The response structure

The full response has two top-level parts: `ResponseMetadata` (HTTP details) and `modelSummaries` (the list of models). Below we show the shape of it: `ResponseMetadata` collapsed, one example model (Nova 2 Lite) with its ARN shown as a placeholder, and the remaining models collapsed. This matches the slide.

In [ ]:
import json
import copy

# Feature one model, like the slide
example = next(
    m for m in response["modelSummaries"] if m["modelId"] == "amazon.nova-2-lite-v1:0"
)
example = copy.deepcopy(example)
example["modelArn"] = "<model-arn>"  # placeholder for brevity

# Collapse the parts the slide abbreviates with ...
view = {
    "ResponseMetadata": "{...}",
    "modelSummaries": [example, "{...}"],
}

print(json.dumps(view, indent=4, default=str))

## 3. Print names and IDs

The `modelId` is the string you pass to a model when you actually invoke it.

In [ ]:
for model in response["modelSummaries"]:
    print(model["modelName"], "-", model["modelId"])

## 4. Look under the hood (DevTools-style view)

boto3 makes a normal HTTPS request to AWS. We can hook into the client to capture the outgoing request and the response it comes back with, then print them like the Headers tab in Chrome DevTools.

Sensitive auth headers are redacted so we don't display live credentials.

In [ ]:
# Headers whose values are secrets - we show that they exist but hide the value
SENSITIVE = {"authorization", "x-amz-security-token"}


def clean_headers(headers):
    """Decode header values to plain strings and redact secrets."""
    result = {}
    for key, value in headers.items():
        if isinstance(value, bytes):
            value = value.decode()
        if key.lower() in SENSITIVE:
            value = "<redacted>"
        result[key] = value
    return result


def show_request(request, **kwargs):
    print("=== REQUEST ===")
    print(request.method, request.url)
    print(json.dumps(clean_headers(request.headers), indent=2))


# Register the hook so it fires right before the request is sent
bedrock.meta.events.register(
    "before-send.bedrock.ListFoundationModels", show_request
)

response = bedrock.list_foundation_models()

print("\n=== RESPONSE ===")
meta = response["ResponseMetadata"]
print("HTTP status:", meta["HTTPStatusCode"])
print(json.dumps(meta["HTTPHeaders"], indent=2))